# Exercise 22: Game Play Analysis IV

**Source:** [LeetCode 550](https://leetcode.com/problems/game-play-analysis-iv/description/?envType=study-plan-v2&envId=top-sql-50) (Medium).

Table: `Activity`
```
+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| player_id    | int     |
| device_id    | int     |
| event_date   | date    |
| games_played | int     |
+--------------+---------+
```
- `(player_id, event_date)` is the primary key.
- Each row is the record of a player `player_id` at playing a number of games `games_played` on a device with `device_id` at the given date `event_date`.

**Task.**
Find the fraction among all players **rounded to 2 decimals** who logged in again the day after the day they first logged in.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_22 WITH (FORCE);
%sql CREATE DATABASE exercise_22;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_22 --alias exercise_22

Connecting and switching to connection 'exercise_22'

#### Fill the database with content

In [4]:
%%sql

CREATE TABLE IF NOT EXISTS Activity (
    player_id INTEGER NOT NULL,
    device_id INTEGER NOT NULL,
    event_date DATE NOT NULL,
    games_played INTEGER,
    PRIMARY KEY (player_id, event_date)
);

INSERT INTO Activity (player_id, device_id, event_date, games_played)
VALUES
(1, 2, '2016-03-01', 5),
(1, 2, '2016-03-02', 6),
(2, 3, '2017-06-25', 1),
(3, 1, '2016-03-02', 0),
(3, 4, '2018-07-03', 5);

Running query in 'exercise_22'

5 rows affected.

++
||
++
++

#### Solve the exercise

**Task.**
Find the fraction among all players **rounded to 2 decimals** who logged in again the day after the day they first logged in.

In [5]:
%%sql

SELECT
    COALESCE(ROUND(COUNT(CASE WHEN A.event_date = B.first_logged_in + INTERVAL '1 day' THEN 1 END)::NUMERIC/COUNT(DISTINCT A.player_id),2), 0) AS fraction
FROM
Activity A
LEFT JOIN
(SELECT
    player_id, MIN(event_date) AS first_logged_in
FROM
    Activity
GROUP BY
    player_id
) B
ON
A.player_id = B.player_id

Running query in 'exercise_22'

1 rows affected.

fraction
0.33


#### Explanation

- In subquery make a table that collects only the first day (`MIN(event_date)`) as `first_logged_in` a player logged in (using `GROUP BY player_id`).
- Perform a `LEFT JOIN` of the `Activity` table with this table on `player_id`.
- `COUNT` all the rows where `first_looged_in` and `event_date` differ by `1` day and divide by the number of players (`COUNT DISTINCT player_id`).
- Return the fraction using `ROUND(_,2)` and `COALESCE(_,0)` in case the table is empty.

#### Close connection

In [6]:
%sql --close exercise_22
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_22;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)